## Examples from *lme4: Mixed-effects Modeling with R* (Bates, 2010)

Walks through the main worked examples in `example/data/lMMwR.pdf` to
demonstrate `lmpy.lme`. Numerical outputs are pinned in
`tests/test_lme_examples.py`.

The model API matches lme4's: `formula` uses bar syntax (`(1|g)` scalar,
`(1+x|g)` correlated vector, `(0+x|g)` zero-intercept slope, `(1|a) +
(1|a:b)` nested), `REML=True` is the default, and printed numerics are
the standard ML/REML summary.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import chi2

from lmpy.lme import lme

DATA = Path('..').resolve() / 'datasets'
def D(pkg, name):
    return pd.read_csv(DATA / pkg / f'{name}.csv')

### Ch 1 § 1.4 — `Dyestuff`: a single random factor

In [2]:
Dyestuff = D('lme4', 'Dyestuff')
fm01 = lme('Yield ~ 1 + (1|Batch)', Dyestuff)              # REML default
print(fm01)
print('REML criterion:', fm01.REML_criterion)

lme[REML]: Yield ~ 1 + (1|Batch)

Random effects:
                   Batch: SD=[42]
                Residual: SD=49.51

Fixed effects:
          (Intercept)
Estimate       1527.5
REML criterion: 319.6542768426505


Same fit by ML — note the deviance / AIC / BIC summary the book prints in Table 1.4.

In [3]:
fm01ML = lme('Yield ~ 1 + (1|Batch)', Dyestuff, REML=False)
print(fm01ML)
print(f'AIC={fm01ML.AIC:.4f}  BIC={fm01ML.BIC:.4f}  '
      f'logLik={fm01ML.loglike:.4f}  deviance={fm01ML.deviance:.4f}  '
      f'df.resid={fm01ML.df_resid}')

lme[ML]: Yield ~ 1 + (1|Batch)

Random effects:
                   Batch: SD=[37.26]
                Residual: SD=49.51

Fixed effects:
          (Intercept)
Estimate       1527.5
AIC=333.3271  BIC=337.5307  logLik=-163.6635  deviance=327.3271  df.resid=27


`Dyestuff2` (§ 1.5) — singular fit: REML drives σ̂_Batch to 0.

In [4]:
Dyestuff2 = D('lme4', 'Dyestuff2')
fm02 = lme('Yield ~ 1 + (1|Batch)', Dyestuff2)
print(fm02)

lme[REML]: Yield ~ 1 + (1|Batch)

Random effects:
                   Batch: SD=[8.542e-08]
                Residual: SD=3.716

Fixed effects:
          (Intercept)
Estimate       5.6656


### Ch 2 § 2.1 — `Penicillin`: crossed random factors

In [5]:
Penicillin = D('lme4', 'Penicillin')
fm03 = lme('diameter ~ 1 + (1|plate) + (1|sample)', Penicillin)
print(fm03)
print('n =', fm03.n, ' n_groups =', fm03.n_groups)

lme[REML]: diameter ~ 1 + (1|plate) + (1|sample)

Random effects:
                   plate: SD=[0.8468]
                  sample: SD=[1.932]
                Residual: SD=0.5499

Fixed effects:
          (Intercept)
Estimate    22.972222
n = 144  n_groups = {'plate': 24, 'sample': 6}


### Ch 2 § 2.2 — `Pastes`: nested random factors and an LRT for σ_batch = 0

`fm04` includes both `(1|sample)` and `(1|batch)`. The reduced model
`fm04a` drops `(1|batch)`. Bates uses `anova(fm04a, fm04)` to test
whether σ_batch is non-zero (book § 2.2.4).

In [6]:
Pastes = D('lme4', 'Pastes')
fm04  = lme('strength ~ 1 + (1|sample) + (1|batch)', Pastes, REML=False)
fm04a = lme('strength ~ 1 + (1|sample)',             Pastes, REML=False)
print(fm04)
print()
print(fm04a)

lme[ML]: strength ~ 1 + (1|sample) + (1|batch)

Random effects:
                  sample: SD=[2.904]
                   batch: SD=[1.095]
                Residual: SD=0.8235

Fixed effects:
          (Intercept)
Estimate    60.053333

lme[ML]: strength ~ 1 + (1|sample)

Random effects:
                  sample: SD=[3.104]
                Residual: SD=0.8234

Fixed effects:
          (Intercept)
Estimate    60.053333


In [7]:
chisq = fm04a.deviance - fm04.deviance
df    = fm04.npar - fm04a.npar
p     = chi2.sf(chisq, df)
print(f'LRT fm04a vs fm04:  chisq={chisq:.4f}  df={df}  p={p:.4f}')

LRT fm04a vs fm04:  chisq=0.4072  df=1  p=0.5234


### Ch 3 § 3.2 — `sleepstudy`: random slopes

`fm07` is the **uncorrelated** version: `(1|Subject) +
(0+Days|Subject)`. Because the same group `Subject` appears in two
bars, the second one shows up under the disambiguated key
`Subject.1`.

In [8]:
sleepstudy = D('lme4', 'sleepstudy')
fm07 = lme(
    'Reaction ~ 1 + Days + (1|Subject) + (0+Days|Subject)',
    sleepstudy, REML=False,
)
print(fm07)

lme[ML]: Reaction ~ 1 + Days + (1|Subject) + (0+Days|Subject)

Random effects:
                 Subject: SD=[24.17]
               Subject.1: SD=[5.799]
                Residual: SD=25.56

Fixed effects:
          (Intercept)       Days
Estimate   251.405105  10.467286


`fm06` is the **correlated** version: `(1+Days|Subject)` — a single
vector bar of size 2, fitting an intercept SD, a slope SD, and one
correlation parameter.

In [9]:
fm06 = lme('Reaction ~ 1 + Days + (1+Days|Subject)', sleepstudy, REML=False)
print(fm06)

lme[ML]: Reaction ~ 1 + Days + (1+Days|Subject)

Random effects:
                 Subject: SD=[23.78, 5.717]
                    corr: [0.081]
                Residual: SD=25.59

Fixed effects:
          (Intercept)       Days
Estimate   251.405105  10.467286


LRT for the correlation: `fm06` adds 1 parameter (the off-diagonal of
the Cholesky factor) over `fm07`. Book result: χ²=0.0639, p=0.8004.

In [10]:
chisq = fm07.deviance - fm06.deviance
df    = fm06.npar - fm07.npar
p     = chi2.sf(chisq, df)
print(f'LRT fm07 vs fm06:  chisq={chisq:.4f}  df={df}  p={p:.4f}')

LRT fm07 vs fm06:  chisq=0.0639  df=1  p=0.8004


### Ch 4 § 4.1 — `Machines`: fixed factor + nested random factor

In [11]:
Machines = D('nlme', 'Machines')
fm10 = lme(
    'score ~ Machine + (1|Worker) + (1|Machine:Worker)',
    Machines, REML=False,
)
print(fm10)
print('n_groups:', fm10.n_groups)

lme[ML]: score ~ Machine + (1|Worker) + (1|Machine:Worker)

Random effects:
          Machine:Worker: SD=[3.396]
                  Worker: SD=[4.364]
                Residual: SD=0.9617

Fixed effects:
          (Intercept)  MachineB   MachineC
Estimate    52.355556  7.966667  13.916667
n_groups: {'Machine:Worker': 18, 'Worker': 6}


### Ch 4 § 4.2 — `ergoStool`: random vs fixed treatment

`fm16` treats both `Subject` and `Type` as random. `fm17` keeps
`Subject` random but treats `Type` (4 levels) as a fixed effect — the
more conventional choice.

In [12]:
ergoStool = D('nlme', 'ergoStool')
fm16 = lme('effort ~ 1 + (1|Subject) + (1|Type)', ergoStool, REML=False)
print(fm16)

lme[ML]: effort ~ 1 + (1|Subject) + (1|Type)

Random effects:
                 Subject: SD=[1.305]
                    Type: SD=[1.505]
                Residual: SD=1.101

Fixed effects:
          (Intercept)
Estimate        10.25


In [13]:
fm17 = lme('effort ~ 1 + Type + (1|Subject)', ergoStool, REML=False)
print(fm17)

lme[ML]: effort ~ 1 + Type + (1|Subject)

Random effects:
                 Subject: SD=[1.256]
                Residual: SD=1.037

Fixed effects:
          (Intercept)    TypeT2    TypeT3    TypeT4
Estimate     8.555556  3.888889  2.222222  0.666667
